# Synthetic hard examples with TA-TiTok + MaskGen
Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all. Upload `train_images.zip` when asked. `synthetic.zip` downloads at the end.

In [ ]:
!nvidia-smi -L
!git clone -q https://github.com/bytedance/1d-tokenizer.git && cd 1d-tokenizer && git checkout -q 942a96fbdd873780179d1b78d5462911528bf8c8
!sed -i 's/pooled_embed, x = text_global_pool(x, idxs, clip_encoder.text_pool_type)/pooled_embed = text_global_pool(x, idxs, clip_encoder.text_pool_type)/' 1d-tokenizer/modeling/maskgen.py
!grep -n "text_global_pool(x" 1d-tokenizer/modeling/maskgen.py
!pip -q install open_clip_torch diffusers omegaconf einops accelerate

In [ ]:
from google.colab import files
import zipfile, shutil
uploaded = files.upload()
shutil.rmtree("train_images", ignore_errors=True)
with zipfile.ZipFile(next(iter(uploaded))) as z:
    z.extractall(".")

In [ ]:
import sys, math, random, csv
from pathlib import Path
import numpy as np
import torch
from PIL import Image, ImageOps
sys.path.insert(0, "1d-tokenizer")
from modeling.tatitok import TATiTok
from modeling.maskgen import MaskGen_VQ, open_clip_text_encoding, get_masking_ratio

SRC_DIR = Path("train_images")
OUT_DIR = Path("synthetic")
SIZE = 256
STEPS = 12
CFG = 6.0
TEMPERATURE = 2.0

NAMES = {"+": "plus sign", "-": "minus sign", "=": "equals sign", "0": "digit zero", "1": "digit one",
         "2": "digit two", "3": "digit three", "4": "digit four", "5": "digit five", "6": "digit six",
         "7": "digit seven", "8": "digit eight", "9": "digit nine"}
STYLES = {
    "thin_pen": "a handwritten {} written with a thin blue ballpoint pen on white paper",
    "marker_bleed": "a handwritten {} drawn with a thick black marker, ink bleeding into the paper",
    "pencil_grid": "a handwritten {} written with a grey pencil on squared notebook paper",
    "blurry_photo": "a blurry phone photo of a handwritten {} in a school notebook",
    "faded_ink": "a faded handwritten {} in light blue ink on old paper",
    "rotated": "a handwritten {} on paper, tilted",
}


def to_square(img):
    w, h = img.size
    s = max(w, h)
    canvas = Image.new("RGB", (s, s), (255, 255, 255))
    canvas.paste(img, ((s - w) // 2, (s - h) // 2))
    return canvas, ((s - w) // 2, (s - h) // 2, (s - w) // 2 + w, (s - h) // 2 + h), s


def prepare(img, rotate=0):
    sq, box, s = to_square(img)
    if rotate:
        sq = sq.rotate(rotate, resample=Image.BICUBIC, fillcolor=(255, 255, 255))
    sq = sq.resize((SIZE, SIZE), Image.BICUBIC)
    x = torch.from_numpy(np.asarray(sq, dtype=np.float32) / 255.0).permute(2, 0, 1)
    return x, box, s


def restore(x, box, s, size):
    arr = (x.clamp(0, 1) * 255).permute(1, 2, 0).to("cpu", torch.uint8).numpy()
    img = Image.fromarray(arr).resize((s, s), Image.BICUBIC).crop(box)
    return img.resize(size, Image.BICUBIC)


def decoder_text(captions):
    idx = clip_tokenizer(captions).to(device)
    cast = clip_encoder.transformer.get_cast_dtype()
    t = clip_encoder.token_embedding(idx).to(cast) + clip_encoder.positional_embedding.to(cast)
    t = clip_encoder.transformer(t.permute(1, 0, 2), attn_mask=clip_encoder.attn_mask).permute(1, 0, 2)
    return clip_encoder.ln_final(t)


def gumbel(t, temperature):
    noise = torch.zeros_like(t).uniform_(0, 1)
    return t + temperature * -torch.log((-torch.log(noise.clamp(min=1e-20))).clamp(min=1e-20))


@torch.no_grad()
def regenerate(tokens, captions, mask_ratio):
    b, n = tokens.shape
    mask_id = generator.mask_token_id
    order = torch.rand(b, n, device=device).argsort(dim=-1)
    initial = order < int(round(n * mask_ratio))
    ids = torch.where(initial, torch.full_like(tokens, mask_id), tokens)
    total = initial.sum(1)
    cond, cond_pooled = open_clip_text_encoding(clip_tokenizer, clip_encoder, captions)
    none, none_pooled = open_clip_text_encoding(clip_tokenizer, clip_encoder, [""])
    none, none_pooled = none.repeat(b, 1, 1), none_pooled.repeat(b, 1, 1)
    aes = torch.full((2 * b,), generator.sample_aesthetic_score, device=device)
    for step in range(STEPS):
        ratio = (step + 1) / STEPS
        is_mask = ids == mask_id
        scale = (CFG - 1) * (1 - math.cos(ratio * math.pi)) / 2 + 1
        logits = generator(torch.cat([ids, ids]), torch.cat([cond, none]), torch.cat([cond_pooled, none_pooled]),
                           aesthetic_score=aes)[0]
        c, u = logits[:b], logits[b:]
        logits = (c + (c - u) * scale) / (0.5 + 0.8 * (1 - ratio))
        temp = TEMPERATURE * (1 - ratio)
        sampled = gumbel(logits, temp).argmax(-1)
        sampled_logits = torch.gather(logits, -1, sampled.unsqueeze(-1)).squeeze(-1)
        sampled = torch.where(is_mask, sampled, ids)
        if step == STEPS - 1:
            ids = sampled
            break
        conf = gumbel(torch.where(is_mask, sampled_logits, torch.full_like(sampled_logits, float("inf"))), temp)
        keep_masked = torch.floor(total.float() * float(get_masking_ratio(ratio, "arccos"))).long()
        keep_masked = torch.minimum(keep_masked, is_mask.sum(1) - 1).clamp(min=0)
        cut = torch.sort(conf, dim=-1)[0].gather(1, (keep_masked - 1).clamp(min=0).unsqueeze(1))
        remask = (conf <= cut) & (keep_masked.unsqueeze(1) > 0)
        ids = torch.where(remask, torch.full_like(sampled, mask_id), sampled)
    return ids


@torch.no_grad()
def make_variants(img, label, seed):
    random.seed(seed)
    torch.manual_seed(seed)
    styles = list(STYLES)
    batch, boxes, ratios = [], [], []
    for style in styles:
        x, box, s = prepare(img, rotate=random.choice([-1, 1]) * random.uniform(12, 25) if style == "rotated" else 0)
        batch.append(x)
        boxes.append((box, s))
    x = torch.stack(batch).to(device)
    tokens = tokenizer.encode(x)[1]["min_encoding_indices"].reshape(len(styles), -1)
    captions = [STYLES[s].format(NAMES[label]) for s in styles]
    ratio = random.uniform(0.35, 0.6)
    ids = regenerate(tokens, captions, ratio)
    images = tokenizer.decode_tokens(ids, decoder_text(captions))
    return [(style, restore(images[i], *boxes[i], img.size), ratio) for i, style in enumerate(styles)]


def run_all():
    OUT_DIR.mkdir(exist_ok=True)
    rows = []
    sources = sorted(p for p in SRC_DIR.rglob("*.png"))
    for k, path in enumerate(sources):
        label = path.parent.name
        img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
        (OUT_DIR / label).mkdir(exist_ok=True)
        for style, out, ratio in make_variants(img, label, seed=k):
            name = f"{label}/{path.stem}_{style}.png"
            out.save(OUT_DIR / name)
            rows.append({"file": name, "label": label, "source": f"{label}/{path.name}", "style": style,
                         "mask_ratio": round(ratio, 3)})
        if k % 25 == 0:
            print(f"{k + 1}/{len(sources)} done", flush=True)
    with open(OUT_DIR / "synthetic.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    return rows

In [ ]:
import open_clip
device = "cuda"
tokenizer = TATiTok.from_pretrained("turkeyju/tokenizer_tatitok_bl128_vq").to(device).eval().requires_grad_(False)
generator = MaskGen_VQ.from_pretrained("turkeyju/generator_maskgen_vq_l").to(device).eval().requires_grad_(False)
clip_encoder, _, _ = open_clip.create_model_and_transforms("ViT-L-14-336", pretrained="openai")
del clip_encoder.visual
clip_tokenizer = open_clip.get_tokenizer("ViT-L-14-336")
clip_encoder.transformer.batch_first = False
clip_encoder = clip_encoder.to(device).eval().requires_grad_(False)

In [ ]:
import matplotlib.pyplot as plt
first = sorted(SRC_DIR.rglob("*.png"))[0]
src = Image.open(first).convert("RGB")
variants = make_variants(src, first.parent.name, seed=0)
fig, ax = plt.subplots(1, len(variants) + 1, figsize=(14, 2.5))
ax[0].imshow(src); ax[0].set_title("real"); ax[0].axis("off")
for a, (style, im, _) in zip(ax[1:], variants):
    a.imshow(im); a.set_title(f"SYNTHETIC\n{style}", fontsize=8); a.axis("off")
plt.show()

In [ ]:
import time
t = time.time()
rows = run_all()
print(len(rows), "images in", round(time.time() - t), "sec")

In [ ]:
shutil.make_archive("synthetic", "zip", ".", "synthetic")
files.download("synthetic.zip")